# `baseline` on a Colab T4 — Grounding DINO + MoGe-2

**FILE PURPOSE** — Runs the `baseline` combination from `poc/combinations.json` on a free
Colab T4.

**Naming convention:** `poc/colab/<combination_id>_colab.ipynb`. The id comes from
`combinations.json`, so `det_sam3` becomes `det_sam3_colab.ipynb`. One notebook per
combination, named after the combination. Scores detection against
HomeObjects-3K ground truth, and produces depth maps for manual review.

## Before you start

**Runtime → Change runtime type → T4 GPU.** The first cell checks this and will tell you
if you forgot.

## What this proves, and what it does not

- **Proves:** which detector finds our furniture, at what precision and recall, and how the
  thresholds trade off. Also whether **doors** are found reliably — the scale anchor
  depends on it.
- **Does NOT prove anything about volume.** HomeObjects-3K has no depth, no dimensions, no
  camera intrinsics. Depth output here is for *looking at*, not scoring. Gap **A1** stays
  open for measurement until you hand-measure real rooms.

## Licence note

HomeObjects-3K is **AGPL-3.0** (Ultralytics). Evaluating on it is ordinary research use.
**Do not train a model you intend to ship on it** — see gap C11. We download the zip
directly and read it with OpenCV, so no Ultralytics code enters the pipeline.

In [ ]:
# ---------------------------------------------------------------------------
# CELL 1 · Check we actually got a GPU
# If this says "cpu", go to Runtime -> Change runtime type -> T4 GPU and rerun.
# ---------------------------------------------------------------------------
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU DETECTED")
import torch
print(f"torch {torch.__version__}  cuda available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# ---------------------------------------------------------------------------
# CELL 2 · Get the code
#
# Option A (default): clone from GitHub. Works once you have pushed.
# Option B: comment the clone and use the file-upload block instead.
# ---------------------------------------------------------------------------
import os
from pathlib import Path

REPO = "https://github.com/ikramHoque/object-with-dimention-detection"

if not Path("/content/repo").exists():
    !git clone --depth 1 {REPO} /content/repo

os.chdir("/content/repo")
print(Path.cwd())
!ls

# --- Option B: no push yet? Upload a zip of the repo instead ---------------
# from google.colab import files
# up = files.upload()                       # pick your zip
# !mkdir -p /content/repo && unzip -q -o $(ls *.zip | head -1) -d /content/repo
# os.chdir("/content/repo")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 3 · Install dependencies
#
# Only what the first attempt needs. NOT ultralytics — that is AGPL-3.0 and is
# deliberately excluded. See MODELS.md section 1b.
# Takes ~3-5 minutes.
# ---------------------------------------------------------------------------
!pip install -q --upgrade transformers timm opencv-python-headless
!pip install -q --no-deps git+https://github.com/microsoft/MoGe.git@b942f00bdc2a2a23ebb474fbe034d487e6dcceec
!pip install -q git+https://github.com/EasternJournalist/utils3d.git@3fab839f0be9931dac7c8488eb0e1600c236e183
# Pinned to MoGe v2.0.0 with --no-deps on purpose. MoGe's main branch is
# MoGe-3, which pulls a cu130 torch wheel and would replace Colab's own
# torch, plus flex-gemm and gradio. We only import moge.model.v2.

import transformers, cv2
print(f"transformers {transformers.__version__}")
print(f"opencv {cv2.__version__}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 4 · Download HomeObjects-3K (~390 MB) and inspect it
#
# Look at `images_with_a_door`. That number is how many test images can exercise
# the scale anchor at all.
# ---------------------------------------------------------------------------
import sys
sys.path.insert(0, "/content/repo")

from poc.datasets import homeobjects as HO
HO.download()

import json
print(json.dumps(HO.summary("val"), indent=2))
print(f"\nScorable prompts ({len(HO.SCORABLE_PROMPTS)} of our 26):")
print(f"  {HO.SCORABLE_PROMPTS}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 5 · The first real number: Grounding DINO scored against ground truth
#
# 50 images. On a T4 this takes roughly 1-2 minutes.
#
# WHAT TO LOOK AT FIRST, in order:
#   1. `door` recall      - if this is low, stage 4 has nothing to anchor on
#   2. `wardrobe` recall  - core removal item, and the hardest to detect
#   3. overall recall     - for a survey, recall matters more than precision
# ---------------------------------------------------------------------------
from poc.runner.run_dataset_eval import run, report

r = run(detector="grounding_dino", limit=50, split="val",
        iou_th=0.5, box_th=0.30, txt_th=0.25,
        depth_key=None, allow_nc=False)
report(r)

In [ ]:
# ---------------------------------------------------------------------------
# CELL 6 · Threshold sweep — see the precision/recall trade with your own eyes
#
# This is the parameter lesson made concrete. Watch precision fall as recall
# rises. Pick the row you want, not the one with the best F1: for a survey a
# missed wardrobe costs far more than a false positive the reviewer deletes.
# ---------------------------------------------------------------------------
rows = []
print(f"  {'box_th':>7} {'precision':>10} {'recall':>8} {'F1':>7} {'fp':>6} {'fn':>6}")
for th in (0.15, 0.20, 0.25, 0.30, 0.40, 0.50):
    rr = run(detector="grounding_dino", limit=50, split="val",
             iou_th=0.5, box_th=th, txt_th=0.25,
             depth_key=None, allow_nc=False, quiet=True)
    rows.append(rr)
    print(f"  {th:>7.2f} {rr['precision']:>10.3f} {rr['recall']:>8.3f} "
          f"{rr['f1']:>7.3f} {rr['counts']['fp']:>6} {rr['counts']['fn']:>6}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 7 · Compare detectors on identical images
#
# Same 50 images, same thresholds, three shippable detectors. This is the
# detector axis of the sweep, run properly.
#
# NOTE rtdetr is CLOSED vocabulary - it has no class for 9 of our 26 prompts, so
# its recall is expected to be lower. That is the point of including it: it
# measures what open vocabulary is worth.
# ---------------------------------------------------------------------------
import pandas as pd

det_rows = []
for d in ["grounding_dino", "owlv2", "rtdetr"]:
    try:
        rr = run(detector=d, limit=50, split="val", iou_th=0.5,
                 box_th=0.30, txt_th=0.25, depth_key=None,
                 allow_nc=False, quiet=True)
        door = rr["per_class"].get("door", {})
        ward = rr["per_class"].get("wardrobe", {})
        det_rows.append(dict(detector=d,
                        precision=rr["precision"], recall=rr["recall"], f1=rr["f1"],
                        door_recall=round(door.get("tp", 0) / door["gt"], 3) if door.get("gt") else None,
                        wardrobe_recall=round(ward.get("tp", 0) / ward["gt"], 3) if ward.get("gt") else None,
                        sec_per_img=rr["sec_per_image"]))
        print(f"  {d} done")
    except Exception as e:
        print(f"  {d} FAILED: {type(e).__name__}: {e}")

pd.DataFrame(det_rows).sort_values("f1", ascending=False)

In [ ]:
# ---------------------------------------------------------------------------
# CELL 8 · Depth, for manual review only
#
# HomeObjects-3K cannot score depth. Look at these and judge by eye:
#   * do object boundaries appear in the depth map where they should?
#   * is the vertical span plausible for a room? (2-3 m floor to ceiling)
#   * is the valid-pixel fraction high? (>80%)
#
# If the vertical span reads like room WIDTH rather than height, the axis
# convention is different from what stage 4 assumes -> flip VERT_AXIS.
# ---------------------------------------------------------------------------
import cv2, numpy as np, matplotlib.pyplot as plt
from poc.models import registry
from poc.runner.pipeline import resize_long

dep = registry.build("moge2")
items = HO.load("val", limit=4)

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for i, it in enumerate(items):
    img = resize_long(cv2.imread(str(it["image"])), 1024)
    g = dep.infer(img)
    m = g.mask.astype(bool)
    axes[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(it["image"].name[:24], fontsize=8); axes[0, i].axis("off")
    dm = g.depth.copy(); dm[~m] = np.nan
    axes[1, i].imshow(dm, cmap="viridis")
    v = g.points[..., g.vert_axis][m]
    axes[1, i].set_title(f"{g.depth[m].min():.1f}-{g.depth[m].max():.1f}m  "
                         f"vspan {v.max()-v.min():.1f}m", fontsize=8)
    axes[1, i].axis("off")
plt.suptitle("MoGe-2 depth — DIAGNOSTIC ONLY, this dataset cannot score it", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# CELL 9 · Detections drawn on the images, next to ground truth
#
# The single most useful cell for manual review. Green = ground truth,
# orange = our prediction. Look for: missed items, doubled boxes, wrong labels.
# ---------------------------------------------------------------------------
from poc.models import registry
from poc.runner.pipeline import resize_long
import json as _json

det = registry.build("grounding_dino", box_threshold=0.25, text_threshold=0.25)
prompts = list(_json.loads(open("poc/detect_vocab.json").read())["prompts"].keys())
items = HO.load("val", limit=6)

fig, axes = plt.subplots(2, 3, figsize=(19, 10))
for ax, it in zip(axes.ravel(), items):
    img = resize_long(cv2.imread(str(it["image"])), 1024)
    H, W = img.shape[:2]
    v = img.copy()
    for b in it["boxes"]:                                    # ground truth, green
        if not b["prompt"]:
            continue
        x0, y0, x1, y1 = b["box_norm"]
        cv2.rectangle(v, (int(x0*W), int(y0*H)), (int(x1*W), int(y1*H)), (80, 220, 80), 3)
        cv2.putText(v, f"GT {b['cls']}", (int(x0*W), max(16, int(y0*H)-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (80, 220, 80), 2)
    for d in det.detect(img, prompts):                       # ours, orange
        x0, y0, x1, y1 = [int(z) for z in d.box]
        cv2.rectangle(v, (x0, y0), (x1, y1), (40, 140, 250), 2)
        cv2.putText(v, f"{d.label} {d.score:.2f}", (x0, min(H-6, y1+16)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (40, 140, 250), 1)
    ax.imshow(cv2.cvtColor(v, cv2.COLOR_BGR2RGB))
    ax.set_title(it["image"].name[:30], fontsize=8); ax.axis("off")
plt.suptitle("GREEN = ground truth   ORANGE = Grounding DINO", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# CELL 10 · Save the results and download them
#
# Colab sessions are wiped when they disconnect. Pull the JSON down and commit
# it locally so the numbers survive.
# ---------------------------------------------------------------------------
import json, datetime, shutil
from pathlib import Path

out = Path("/content/repo/poc/results"); out.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

payload = dict(
    dataset="HomeObjects-3K",
    dataset_licence="AGPL-3.0 (Ultralytics) — evaluation only; "
                    "do not train a shippable model on it (gap C11)",
    scores_detection_only=True,
    note="No depth/dimension/volume ground truth in this dataset. "
         "Gap A1 remains open for measurement.",
    baseline=r,                                    # cell 5
    threshold_sweep=rows,                          # cell 6
    detector_comparison=det_rows,                  # cell 7
)

f = out / f"baseline_colab__{stamp}.json"
f.write_text(json.dumps(payload, indent=2, default=str))
print(f"wrote {f}")

shutil.make_archive("/content/results", "zip", out)
from google.colab import files
files.download("/content/results.zip")

## What to do with these results

1. **`door` recall is the first thing to check.** The scale anchor cannot work if doors
   aren't found. If recall is under ~0.6, either lower `box_th`, or accept that the anchor
   needs a fallback (ask the customer to confirm room size — gap A5 option (c)).

2. **`wardrobe` recall is the second.** It's a core removal item and the hardest class here.
   Poor recall means the vocabulary or the threshold needs work before any volume number is
   worth computing.

3. **Pick a `box_th` from the sweep, not from the best F1.** For a survey, recall wins:
   a missed wardrobe is a 100% error on that item, a false positive costs the reviewer two
   seconds.

4. **Then commit the JSON locally** and record what you found in `STATUS.md`.

## What still isn't answered

This tells you nothing about **volume**, because the dataset has no dimensions. To close
that you need the tape-measure step — gap **A1**. Three to five real rooms, hand measured.
That remains the highest-value half-day in the project.